# Epithelial reintegration, specimen origin and CNV sensitivity

## Imports


In [ ]:
from __future__ import annotations
from pathlib import Path
import argparse
import ast
import gc

from tensorboard import notebook

from cap.figures import (
    configure_figure_style,
    save_figure,
    save_umap_figure,
)
from cap.paths import get_paths
from cap.provenance import (
    sha256_file,
    validate_frozen_inputs,
    write_run_provenance,
)


In [ ]:
import hashlib
import itertools
import json
import logging
import os


In [ ]:
import re
import sys


In [ ]:
import shutil
import subprocess

In [ ]:
from matplotlib import colormaps
from matplotlib.colors import TwoSlopeNorm
from matplotlib.colors import to_hex
from matplotlib.lines import Line2D
from scipy import stats


In [ ]:
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from statsmodels.stats.multitest import multipletests


In [ ]:
import anndata as ad
import h5py
import infercnvpy as cnv
import matplotlib.pyplot as plt
import numpy as np


In [ ]:
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import scvi
import seaborn as sns

In [ ]:
# Configure R before importing rpy2.
r_home_value = os.environ.get("R_HOME")
if not r_home_value:
    raise RuntimeError(
        "R_HOME must be set before starting the notebook kernel."
    )

r_home = Path(r_home_value).expanduser()
if not r_home.is_dir():
    raise RuntimeError("The configured R_HOME is not an existing directory.")

if os.name == "nt":
    r_executables = (
        r_home / "bin" / "x64" / "R.exe",
        r_home / "bin" / "R.exe",
    )
    if not any(path.is_file() for path in r_executables):
        raise RuntimeError("R.exe was not found under the configured R_HOME.")

os.environ["R_HOME"] = str(r_home.resolve())
os.environ.setdefault("RPY2_CFFI_MODE", "ABI")

In [ ]:
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
from rpy2.robjects.packages import importr
import session_info

In [ ]:
import logging
logging.getLogger("fontTools.subset").setLevel(logging.WARNING)

## Paths


In [ ]:
paths = get_paths()
paths.validate_base_inputs()
validate_frozen_inputs(paths)

project = paths.root
analysis_dir = paths.run_root
data_dir = paths.data_root

annotation_dir = paths.stage_dir("02_cell_annotation")
paths.require_dir(annotation_dir, "02_cell_annotation stage")

epithelial_file = annotation_dir / "h5ad" / "epithelial.h5ad"
paths.require_file(epithelial_file, "002 manually annotated epithelial object")

meta_programme_file = paths.frozen_dir / "cancer_meta_programme_signatures.xlsx"
gencode_file = paths.gencode_file
policy_file = paths.frozen_dir / "gene_policy.json"

for path in (meta_programme_file, gencode_file, policy_file):
    paths.require_file(path, path.name)

In [ ]:
policy_sha256 = "c4213d53e2d3209312e7781c73ed20f660e0f256042956742693c0e0375926a6"


In [ ]:
def load_gene_policy(path=policy_file):
    path = Path(path)
    payload = path.read_bytes()
    digest = hashlib.sha256(payload).hexdigest()
    if digest != policy_sha256:
        raise ValueError(
            "Gene-policy checksum differs from the reviewed source snapshot"
        )
    policy = json.loads(payload)
    policy["sha256"] = digest
    return policy

gene_policy = load_gene_policy()

In [ ]:
def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


In [ ]:
assert file_sha256(gencode_file) == gene_policy["gencode_v50_sha256"]

In [ ]:
manifest_file = paths.manifest_dir / "frozen_inputs.json"
paths.require_file(manifest_file, "frozen_inputs.json")

# Calculate input checksums before creating or writing to the output stage.
input_checksums = {
    "02_cell_annotation/h5ad/epithelial.h5ad": sha256_file(epithelial_file),
    "gene_policy.json": sha256_file(policy_file),
    "cancer_meta_programme_signatures.xlsx": sha256_file(meta_programme_file),
    "gencode.v50.annotation.gtf.gz": sha256_file(gencode_file),
    "frozen_inputs.json": sha256_file(manifest_file),
}

In [ ]:
STAGE = "03a_epithelial_integration"
notebook_path = paths.root / "notebooks" / "003a_Epithelial_Integration.ipynb"
paths.require_file(notebook_path, "003a notebook source")

input_checksums = {
    "02_cell_annotation/h5ad/epithelial.h5ad": sha256_file(epithelial_file),
    "gene_policy.json": sha256_file(paths.frozen_dir / "gene_policy.json"),
    "cancer_meta_programme_signatures.xlsx": sha256_file(
        paths.frozen_dir / "cancer_meta_programme_signatures.xlsx"
    ),
    "gencode.v50.annotation.gtf.gz": sha256_file(paths.gencode_file),
    "frozen_inputs.json": sha256_file(paths.manifest_dir / "frozen_inputs.json"),
}

run_dir = paths.prepare_stage(STAGE)   # refuses a non-empty stage

fig_dir = run_dir / "figures"
df_dir = run_dir / "tables"
save_dir = run_dir / "h5ad"
models_dir = run_dir / "models"
model_dir = models_dir / "scvi_epithelial_model"
for directory in (fig_dir, df_dir, save_dir, models_dir):
    directory.mkdir(parents=True, exist_ok=True)

(run_dir / "input_checksums.json").write_text(
    json.dumps(input_checksums, indent=2) + "\n", encoding="utf-8"
)
write_run_provenance(paths, run_dir,
    notebook_path=notebook_path,
    input_checksums=input_checksums,
)

In [ ]:
# CPU setting used for the portable Windows rerun.
scvi_accelerator = "cpu"
np.random.seed(0)
random_seed = 0
scvi.settings.seed = 0
scvi.settings.num_workers = 4
sc.settings.verbosity = 2
rcb.logger.setLevel(logging.ERROR)
configure_figure_style()


In [ ]:
target_genes = ["ROR1", "ROR2", "EGFR", "ERBB2", "EPCAM"]


In [ ]:
n_hvg_primary = 4000
cnv_high_quantile = 0.95
infercnv_dynamic_threshold = 1.5
cnv_quantiles = (0.9, 0.95, 0.99)
n_latent = 20
n_layers = 2
n_neighbors = 30
primary_resolution = 0.5
leiden_resolutions = (0.5, 1.0, 1.2, 1.5)


In [ ]:
gene_likelihood = "nb"
sc.settings.figdir = fig_dir


In [ ]:
def rel(path):
    """Display repository or external-data paths without embedding local absolute paths."""
    path = Path(path).resolve()
    for base, label in ((paths.root, Path(".")), (paths.data_root, Path("CAP_DATA_ROOT"))):
        try:
            return label / path.relative_to(base)
        except ValueError:
            continue
    return Path(path.name)


## Load the epithelial compartment

In [ ]:
epi = sc.read_h5ad(epithelial_file)


In [ ]:
assert "cohort_selection" in epi.uns, "Input lacks the cohort-selection record; use the output from the updated upstream notebook."


In [ ]:
cohort_selection = epi.uns["cohort_selection"]


In [ ]:
assert not epi.obs["patient"].astype(str).isin(
    list(cohort_selection["excluded_patients"])
).any(), "An excluded donor is present in the analysis input."


In [ ]:
assert epi.obs["sample"].nunique() == cohort_selection["n_selected_libraries"], "Library count differs from the upstream cohort-selection record."


In [ ]:
legacy_celltypist_columns = {
    "celltypist_pal_label": "celltypist_reference_label",
    "celltypist_pal_conf": "celltypist_reference_confidence",
}


In [ ]:
assert (
    epi.uns.get("annotation_mapping", {}).get("method") == "manual"
), "Use the epithelial object exported after manual annotation in 002."


In [ ]:
assert epi.obs["cl_annotation"].notna().all()


In [ ]:
assert epi.obs[
    "epithelial_primary"
].all(), "Input contains cells excluded from the epithelial population."


In [ ]:
def feature_identities(var, policy):
    """Assign frozen HGNC IDs by stable matrix IDs, without changing var_names."""
    if not var.index.is_unique or "gene_id" not in var:
        raise ValueError(
            "Unique feature names and original Ensembl gene_id values are required"
        )
    ids = var["gene_id"].astype(str).str.split(".").str[0]
    canonical = {}
    for records in policy["symbols"].values():
        for record in records:
            if record["ensembl_id"]:
                canonical.setdefault(record["ensembl_id"], set()).add(record["hgnc_id"])
    values = []
    for identifier in ids:
        identity = policy["feature_hgnc_ids"].get(identifier)
        if identity is None:
            candidates = canonical.get(identifier, set())
            identity = next(iter(candidates)) if len(candidates) == 1 else ""
        values.append(identity)
    return pd.Series(values, index=var.index, name="hgnc_id")


In [ ]:
def apply_gene_flags(adata, policy):
    """Replace broad QC prefixes with reviewed family memberships."""
    identity = feature_identities(adata.var, policy)
    audit = pd.DataFrame(
        {
            "feature_gene": adata.var_names.astype(str),
            "feature_id": adata.var["gene_id"].astype(str).to_numpy(),
            "hgnc_id": identity.to_numpy(),
        }
    )
    for flag in ["mt", "ribo", "hb"]:
        old = (
            adata.var[flag].astype(bool).to_numpy()
            if flag in adata.var
            else np.zeros(adata.n_vars, bool)
        )
        new = identity.isin(policy["families"][flag]).to_numpy()
        adata.var[flag] = new
        audit[flag + "_before"] = old
        audit[flag + "_after"] = new
    return audit


In [ ]:
def require_gene_policy(adata, policy, upstream):
    record = adata.uns.get("gene_policy", {})
    if record.get("sha256") != policy["sha256"] or not record.get(
        "cell_cycle_scored", False
    ):
        raise ValueError(
            f"Input uses superseded gene definitions; rerun {upstream} with the current gene policy"
        )


In [ ]:
def stamp_gene_policy(adata, policy, stage, cell_cycle_scored=None):
    previous = dict(adata.uns.get("gene_policy", {}))
    previous.update(
        {"version": policy["version"], "sha256": policy["sha256"], "stage": stage}
    )
    if cell_cycle_scored is not None:
        previous["cell_cycle_scored"] = bool(cell_cycle_scored)
    adata.uns["gene_policy"] = previous


In [ ]:
require_gene_policy(epi, gene_policy, "002")


In [ ]:
gene_flag_audit = apply_gene_flags(epi, gene_policy)


In [ ]:
gene_flag_audit.to_csv(df_dir / "epithelial_gene_flags.csv", index=False)


In [ ]:
stamp_gene_policy(epi, gene_policy, "epithelial_integration")


In [ ]:
epi.obs.rename(
    columns={
        old: new
        for old, new in legacy_celltypist_columns.items()
        if old in epi.obs.columns and new not in epi.obs.columns
    },
    inplace=True,
)


In [ ]:
display(
    pd.DataFrame(
        {
            "cells": [epi.n_obs],
            "genes": [epi.n_vars],
            "samples": [epi.obs["sample"].nunique()],
        }
    )
)


In [ ]:
if "X_umap" in epi.obsm and "X_umap_atlas" not in epi.obsm:
    epi.obsm["X_umap_atlas"] = epi.obsm["X_umap"].copy()


In [ ]:
counts = epi.layers["counts"]


In [ ]:
count_values = counts.data if sp.issparse(counts) else np.asarray(counts).ravel()


In [ ]:
assert np.isfinite(count_values).all()


In [ ]:
assert np.allclose(count_values, np.rint(count_values))


In [ ]:
baseline_positive = {
    gene: int((np.asarray(epi[:, gene].layers["counts"].todense()).ravel() > 0).sum())
    for gene in target_genes
}


In [ ]:
display(pd.Series(baseline_positive, name="cells_with_raw_count"))


## Gene filtering and normalisation

In [ ]:
n_cells_loaded = epi.n_obs


In [ ]:
n_genes_loaded = epi.n_vars


In [ ]:
sc.pp.filter_genes(epi, min_cells=3)


In [ ]:
n_genes_removed = n_genes_loaded - epi.n_vars


In [ ]:
n_cells_start = epi.n_obs


In [ ]:
n_genes_start = epi.n_vars


In [ ]:
shape_summary = pd.DataFrame(
    {
        "stage": ["loaded", "analysis object"],
        "cells": [n_cells_loaded, epi.n_obs],
        "genes": [n_genes_loaded, epi.n_vars],
    }
)


In [ ]:
display(shape_summary)


In [ ]:
epi.layers["lognorm"] = epi.layers["log1p"].copy()


In [ ]:
epi.X = epi.layers["lognorm"].copy()


In [ ]:
n_cells_start = epi.n_obs


In [ ]:
n_genes_start = epi.n_vars


In [ ]:
assert n_cells_start == n_cells_loaded


In [ ]:
assert n_genes_start == n_genes_loaded - n_genes_removed


## Highly variable genes

Binomial deviance (`scry`) computed per sample and aggregated by the maximum
across samples, so that a gene deviant in a single condition is retained.
Mitochondrial, ribosomal and haemoglobin genes are excluded before ranking.
Cell-cycle genes are deliberately **not** excluded here: proliferation is a
tumour-associated epithelial state of interest.

In [ ]:
%load_ext rpy2.ipython


In [ ]:
%%R
.libPaths(unique(c(Sys.getenv("R_LIBS_USER"), .libPaths())))
suppressPackageStartupMessages({
    library(Matrix)
    library(SingleCellExperiment)
    library(scry)})

In [ ]:
def scipy_to_dgc(matrix):
    matrix = sp.csc_matrix(matrix)
    return ro.r["new"](
        "dgCMatrix",
        i=ro.IntVector(matrix.indices),
        p=ro.IntVector(matrix.indptr),
        x=ro.FloatVector(matrix.data),
        Dim=ro.IntVector(matrix.shape),
    )


In [ ]:
samples = epi.obs["sample"].astype(str).unique().tolist()


In [ ]:
deviance = pd.DataFrame(index=epi.var_names, columns=samples, dtype=float)


In [ ]:
for sample in samples:
    sub = epi[epi.obs["sample"].astype(str) == sample]
    ro.globalenv["raw_counts"] = scipy_to_dgc(sub.layers["counts"].T)
    ro.globalenv["gene_names"] = ro.StrVector(sub.var_names.tolist())
    ro.globalenv["cell_names"] = ro.StrVector(sub.obs_names.tolist())
    ro.r(
        """
        rownames(raw_counts) <- gene_names
        colnames(raw_counts) <- cell_names

        sce <- SingleCellExperiment(assays = list(counts = raw_counts))
        sce <- devianceFeatureSelection(sce)

        binomial_deviance <- rowData(sce)$binomial_deviance
    """
    )
    deviance[sample] = np.asarray(ro.r["binomial_deviance"])
    ro.r("rm(raw_counts, sce, binomial_deviance); gc()")
    print(f"{sample}: {sub.n_obs:,} cells")


In [ ]:
epi.var["binomial_deviance"] = deviance.max(axis=1)


In [ ]:
assert epi.var["binomial_deviance"].notna().all()


In [ ]:
for column in ["mt", "ribo", "hb"]:
    assert column in epi.var.columns, column


In [ ]:
exclude = epi.var[["mt", "ribo", "hb"]].any(axis=1)


In [ ]:
exclude |= feature_identities(epi.var, gene_policy).isin(
    gene_policy["families"]["immunoglobulin"]
)


In [ ]:
ranked = epi.var.loc[~exclude, "binomial_deviance"].sort_values(ascending=False).index


In [ ]:
deviant_genes = ranked[:n_hvg_primary]


In [ ]:
meta_programme_sheet = pd.read_excel(meta_programme_file, sheet_name="Cancer MPs")


In [ ]:
meta_programme_genes = (
    meta_programme_sheet.stack()
    .dropna()
    .astype(str)
    .str.strip()
    .loc[lambda x: x.ne("")]
    .unique()
)


In [ ]:
def resolve_signature(genes, var, policy):
    """Return unique available feature names and a complete source-member audit.

    Exact original features remain preferred when consistent with the frozen
    identity. Otherwise a unique stable-ID/HGNC match is required. Ambiguities
    stop the run; absent source members remain explicit audit rows.
    """
    identities = feature_identities(var, policy)
    ids = var["gene_id"].astype(str).str.split(".").str[0]
    rows, selected = ([], [])
    for gene in dict.fromkeys(map(str, genes)):
        candidates = policy["symbols"].get(gene, [])
        hgnc_ids = {r["hgnc_id"] for r in candidates}
        expected_ids = {r["ensembl_id"] for r in candidates if r["ensembl_id"]}
        chosen, method = (None, "unavailable")
        if gene in var.index:
            identity = identities.loc[gene]
            if hgnc_ids and identity and (identity not in hgnc_ids):
                raise ValueError(
                    f"Exact feature conflicts with source identity: {gene}"
                )
            chosen, method = (gene, "exact_original_feature")
        else:
            direct = var.index[ids.isin(expected_ids)]
            matches = direct if len(direct) else var.index[identities.isin(hgnc_ids)]
            if len(matches) > 1:
                raise ValueError(
                    f"Ambiguous source-to-feature mapping: {gene}: {list(matches)}"
                )
            if len(matches) == 1:
                chosen = str(matches[0])
                method = (
                    "verified_Ensembl_ID" if len(direct) else "frozen_HGNC_identity"
                )
        duplicate = chosen is not None and chosen in selected
        if chosen is not None and (not duplicate):
            selected.append(chosen)
        rows.append(
            {
                "source_gene": gene,
                "feature_gene": chosen or "",
                "feature_id": str(ids.loc[chosen]) if chosen is not None else "",
                "hgnc_id": identities.loc[chosen] if chosen is not None else "",
                "method": (
                    "duplicate_feature_not_double_weighted" if duplicate else method
                ),
                "used": chosen is not None and (not duplicate),
            }
        )
    return (selected, pd.DataFrame(rows))


In [ ]:
protected = sorted(set(target_genes) | set(meta_programme_genes))


In [ ]:
protected_present, protected_mapping = resolve_signature(
    protected, epi.var, gene_policy
)


In [ ]:
protected_mapping.to_csv(df_dir / "epithelial_protected_gene_mapping.csv", index=False)


In [ ]:
epi.var["highly_variable_deviance"] = epi.var_names.isin(deviant_genes)


In [ ]:
epi.var["highly_variable_protected"] = epi.var[
    "highly_variable_deviance"
] | epi.var_names.isin(protected_present)


In [ ]:
epi.var["highly_variable"] = epi.var["highly_variable_deviance"].copy()


In [ ]:
n_hvg_deviance = int(epi.var["highly_variable_deviance"].sum())


In [ ]:
n_hvg_protected = int(epi.var["highly_variable_protected"].sum())


In [ ]:
n_protected_added = n_hvg_protected - n_hvg_deviance


In [ ]:
hvg_summary = pd.DataFrame(
    {
        "feature_set": ["primary_deviance", "protected_sensitivity"],
        "n_genes": [n_hvg_deviance, n_hvg_protected],
    }
)


In [ ]:
display(hvg_summary)


In [ ]:
pd.Series(deviant_genes, name="gene").to_csv(
    df_dir / "epithelial_hvg_genes.csv", index=False
)


## inferCNV

Run inferCNV on a separate object so its gene filtering cannot alter the integration object. Genomic coordinates come from GENCODE v50 and are joined by stable Ensembl gene ID. The primary run uses infercnvpy's documented default dynamic threshold of 1.5. The normal-reference quantile sensitivity below does not test this smoothing parameter.

In [ ]:
cnv_adata = ad.AnnData(
    X=epi.layers["lognorm"].copy(), obs=epi.obs.copy(), var=epi.var.copy()
)


In [ ]:
print(
    f"inferCNV working object: {cnv_adata.n_obs:,} cells x {cnv_adata.n_vars:,} genes"
)


In [ ]:
gtf = pd.read_csv(
    gencode_file,
    sep="\t",
    comment="#",
    header=None,
    names=[
        "chromosome",
        "source",
        "feature",
        "start",
        "end",
        "score",
        "strand",
        "frame",
        "attribute",
    ],
)


In [ ]:
gencode = gtf[gtf["feature"] == "gene"].copy()


In [ ]:
gencode["ensembl"] = (
    gencode["attribute"].str.extract('gene_id "([^"]+)"')[0].str.split(".").str[0]
)


In [ ]:
gencode["gene_type"] = gencode["attribute"].str.extract('gene_type "([^"]+)"')[0]


In [ ]:
gencode = gencode.drop_duplicates("ensembl").set_index("ensembl")


In [ ]:
len(gencode)


In [ ]:
del gtf


In [ ]:
gc.collect()


In [ ]:
def cnv_annotation_ids(var, gencode, policy):
    """Apply only the explicitly reviewed historical SOD2 coordinate override."""
    ids = var["gene_id"].astype(str).str.split(".").str[0].copy()
    audit = []
    for item in policy["coordinate_overrides"]:
        target = item["annotation_ensembl_id"]
        if target not in gencode.index:
            raise ValueError(f"Reviewed annotation target absent: {target}")
        record = gencode.loc[target]
        if (
            record.chromosome != item["chromosome"]
            or int(record.start) != item["start"]
            or int(record.end) != item["end"]
            or (record.gene_type != "protein_coding")
        ):
            raise ValueError(
                "SOD2 coordinate override does not match the reviewed GENCODE version"
            )
        mask = ids.eq(item["source_ensembl_id"])
        ids.loc[mask] = target
        for feature in var.index[mask]:
            audit.append({"feature_gene": str(feature), **item})
    return (ids, pd.DataFrame(audit))


In [ ]:
ensembl, coordinate_audit = cnv_annotation_ids(cnv_adata.var, gencode, gene_policy)


In [ ]:
cnv_adata.var["cnv_annotation_id"] = ensembl.to_numpy()


In [ ]:
coordinate_audit.to_csv(df_dir / "infercnv_coordinate_overrides.csv", index=False)


In [ ]:
cnv_adata.var["chromosome"] = ensembl.map(gencode["chromosome"]).values


In [ ]:
cnv_adata.var["start"] = ensembl.map(gencode["start"]).values


In [ ]:
cnv_adata.var["end"] = ensembl.map(gencode["end"]).values


In [ ]:
cnv_adata.var["gene_type"] = ensembl.map(gencode["gene_type"]).values


In [ ]:
resolved = cnv_adata.var["chromosome"].notna()


In [ ]:
del gencode


In [ ]:
gc.collect()


In [ ]:
(resolved.sum(), cnv_adata.n_vars)


In [ ]:
cnv_adata.var.loc[~resolved, ["gene_id"]].to_csv(
    df_dir / "infercnv_unresolved_genes.csv"
)


In [ ]:
autosomes = [f"chr{i}" for i in range(1, 23)]


In [ ]:
keep = cnv_adata.var["chromosome"].isin(autosomes) & cnv_adata.var["gene_type"].eq(
    "protein_coding"
)


In [ ]:
cnv_adata = cnv_adata[:, keep].copy()


In [ ]:
cnv_adata.n_vars


In [ ]:
def cnv_family_mask(var, policy, relaxed=False):
    """Optional relaxation retains the HLA/keratin/SPRR/S100A families for sensitivity."""
    identity = feature_identities(var, policy)
    if relaxed:
        excluded = set(policy["families"]["ribo"]) | set(
            policy["families"]["immunoglobulin"]
        )
    else:
        excluded = set(policy["families"]["cnv_expression_confounds"])
    return identity.isin(excluded).to_numpy()


In [ ]:
problematic = cnv_family_mask(cnv_adata.var, gene_policy)


In [ ]:
family_audit = pd.DataFrame(
    {
        "gene": cnv_adata.var_names,
        "gene_id": cnv_adata.var["gene_id"].to_numpy(),
        "excluded_primary": problematic,
        "excluded_relaxed": cnv_family_mask(cnv_adata.var, gene_policy, relaxed=True),
    }
)


In [ ]:
family_audit.to_csv(df_dir / "infercnv_family_filter_audit.csv", index=False)


In [ ]:
problematic.sum()


In [ ]:
cnv_adata = cnv_adata[:, ~problematic].copy()


In [ ]:
sc.pp.filter_genes(cnv_adata, min_cells=100)


In [ ]:
cnv_adata.n_vars


In [ ]:
cnv_adata.var["chromosome"] = cnv_adata.var["chromosome"].astype(str)


In [ ]:
cnv_adata.var["chr_num"] = (
    cnv_adata.var["chromosome"].str.replace("chr", "", regex=False).astype(int)
)


In [ ]:
order = cnv_adata.var.sort_values(["chr_num", "start"]).index


In [ ]:
cnv_adata = cnv_adata[:, order].copy()


In [ ]:
assert cnv_adata.var_names.is_unique


In [ ]:
assert (np.diff(cnv_adata.var["chr_num"].to_numpy()) >= 0).all()


In [ ]:
reference = cnv_adata[cnv_adata.obs["condition"] == "N"].copy()


In [ ]:
print(
    f"reference cells: {reference.n_obs:,} from {reference.obs['sample'].nunique()} samples"
)


In [ ]:
cnv_adata.obs["cnv_score"] = np.nan


In [ ]:
cnv_adata.obs["cnv_signal"] = np.nan


In [ ]:
x_cnv_full = None


In [ ]:
failures = []


In [ ]:
cnv_samples = cnv_adata.obs["sample"].astype(str).unique().tolist()


In [ ]:
len(cnv_samples)


In [ ]:
for sample in cnv_samples:
    print(f"\n{sample}")
    target = refs = combined = None
    try:
        target = cnv_adata[cnv_adata.obs["sample"].astype(str) == sample].copy()
        condition = str(target.obs["condition"].iloc[0])
        if condition == "N":
            refs = reference[reference.obs["sample"].astype(str) != sample].copy()
        else:
            refs = reference.copy()
        assert refs.n_obs > 0
        refs.obs["cnv_reference"] = "reference"
        target.obs["cnv_reference"] = "query"
        combined = sc.concat([refs, target], join="inner", merge="same")
        print(f"  {condition}: {target.n_obs:,} query, {refs.n_obs:,} reference")
        cnv.tl.infercnv(
            combined,
            reference_key="cnv_reference",
            reference_cat=["reference"],
            window_size=100,
            step=10,
            dynamic_threshold=infercnv_dynamic_threshold,
            exclude_chromosomes=("chrX", "chrY"),
            n_jobs=1,
        )
        x_cnv = combined.obsm["X_cnv"]
        x_cnv = x_cnv.toarray() if sp.issparse(x_cnv) else np.asarray(x_cnv)
        x_cnv = x_cnv.astype(np.float32)
        query = combined.obs["cnv_reference"].eq("query").to_numpy()
        query_names = combined.obs_names[query]
        x_query = x_cnv[query]
        chr_pos = sorted(combined.uns["cnv"]["chr_pos"].items(), key=lambda kv: kv[1])
        n_windows = x_query.shape[1]
        chr_labels = np.empty(n_windows, dtype=object)
        for i, (chrom, start) in enumerate(chr_pos):
            end = chr_pos[i + 1][1] if i + 1 < len(chr_pos) else n_windows
            chr_labels[start:end] = chrom
        chrom_scores = []
        chrom_weights = []
        for chrom in pd.unique(chr_labels):
            idx = np.flatnonzero(chr_labels == chrom)
            chrom_scores.append(np.sqrt(np.mean(x_query[:, idx] ** 2, axis=1)))
            chrom_weights.append(len(idx))
        chrom_scores = np.column_stack(chrom_scores)
        chrom_weights = np.asarray(chrom_weights, dtype=float)
        chrom_weights /= chrom_weights.sum()
        cnv_score = np.average(chrom_scores, axis=1, weights=chrom_weights)
        cnv_signal = np.max(np.abs(x_query), axis=1)
        cnv_adata.obs.loc[query_names, "cnv_score"] = cnv_score
        cnv_adata.obs.loc[query_names, "cnv_signal"] = cnv_signal
        if x_cnv_full is None:
            x_cnv_full = np.full((cnv_adata.n_obs, n_windows), np.nan, dtype=np.float32)
        if "cnv" not in cnv_adata.uns:
            cnv_adata.uns["cnv"] = dict(combined.uns["cnv"])
        x_cnv_full[cnv_adata.obs_names.get_indexer(query_names)] = x_query
        print(f"  median burden {np.median(cnv_score):.4f}")
    except Exception as error:
        print(f"  failed: {error}")
        failures.append({"sample": sample, "error": str(error)})
    finally:
        del target, refs, combined
        gc.collect()


In [ ]:
failures = pd.DataFrame(failures)


In [ ]:
if not failures.empty:
    failures.to_csv(df_dir / "infercnv_failed_samples.csv", index=False)
    raise RuntimeError(f"inferCNV failed for {len(failures)} sample(s)")


In [ ]:
assert cnv_adata.obs[["cnv_score", "cnv_signal"]].notna().all().all()


In [ ]:
display(pd.DataFrame({"infercnv_samples_completed": [len(cnv_samples)]}))


In [ ]:
valid = cnv_adata.obs[["cnv_score", "total_counts"]].dropna()


In [ ]:
r = np.corrcoef(valid["cnv_score"], valid["total_counts"])[0, 1]


In [ ]:
display(pd.DataFrame({"cnv_score_vs_total_counts_r": [r]}).round(3))


In [ ]:
cnv_adata.obs.groupby("condition", observed=True)["cnv_score"].describe().round(4)


## Specimen origin and CNV-high sensitivity

The primary population definition follows specimen origin. Epithelial cells
from ER+, HER2+ and TN specimens are labelled `tumour_derived`; this does not
imply that every cell is malignant. Normal-tissue epithelial cells provide the
reference population.

`cnv_high` is a technical flag applied to every epithelial cell whose inferred
CNV burden exceeds the 95th percentile of normal epithelium. The conjunction
of tumour origin and this flag is stored as `tumour_derived_cnv_high`. This is a
conservative CNV-enriched subgroup, not a classification of every tumour-derived cell.

G1-restricted summaries assess whether cell-cycle transcription
materially affects the CNV-high sensitivity analysis.

In [ ]:
condition_values = cnv_adata.obs["condition"].astype(str)


In [ ]:
phase_values = cnv_adata.obs["phase"].astype(str)


In [ ]:
normal_mask = condition_values.eq("N")


In [ ]:
normal_scores = cnv_adata.obs.loc[normal_mask, "cnv_score"]


In [ ]:
normal_g1_scores = cnv_adata.obs.loc[normal_mask & phase_values.eq("G1"), "cnv_score"]


In [ ]:
cnv_high_threshold = float(normal_scores.quantile(cnv_high_quantile))


In [ ]:
cnv_high_threshold_g1 = float(normal_g1_scores.quantile(cnv_high_quantile))


In [ ]:
cnv_adata.obs["tumour_derived"] = condition_values.ne("N")


In [ ]:
cnv_adata.obs["cnv_high"] = cnv_adata.obs["cnv_score"].gt(cnv_high_threshold)


In [ ]:
cnv_adata.obs["tumour_derived_cnv_high"] = (
    cnv_adata.obs["tumour_derived"] & cnv_adata.obs["cnv_high"]
)


In [ ]:
cnv_adata.obs["g1_cell"] = phase_values.eq("G1")


In [ ]:
cnv_adata.obs["tumour_derived_cnv_high_g1"] = (
    cnv_adata.obs["tumour_derived_cnv_high"] & cnv_adata.obs["g1_cell"]
)


In [ ]:
origin_call = np.where(
    condition_values.eq("N"), "normal-tissue epithelium", "tumour-derived epithelium"
)


In [ ]:
origin_order = ["normal-tissue epithelium", "tumour-derived epithelium"]


In [ ]:
cnv_adata.obs["specimen_origin"] = pd.Categorical(origin_call, categories=origin_order)


In [ ]:
population_counts = pd.crosstab(
    cnv_adata.obs["condition"], cnv_adata.obs["specimen_origin"], margins=True
)


In [ ]:
display(population_counts)


In [ ]:
rows = []


In [ ]:
for quantile in cnv_quantiles:
    threshold = float(normal_scores.quantile(quantile))
    called = cnv_adata.obs["cnv_score"].gt(threshold) & cnv_adata.obs["condition"].ne(
        "N"
    )
    for condition in ["ER+", "HER2+", "TN"]:
        mask = cnv_adata.obs["condition"].eq(condition)
        rows.append(
            {
                "normal_quantile": quantile,
                "threshold": threshold,
                "condition": condition,
                "n_cells": int(mask.sum()),
                "n_cnv_high": int(called[mask].sum()),
                "cnv_high_fraction": float(called[mask].mean()),
            }
        )


In [ ]:
threshold_sensitivity = pd.DataFrame(rows)


In [ ]:
threshold_sensitivity.to_csv(df_dir / "infercnv_threshold_sensitivity.csv", index=False)


In [ ]:
display(threshold_sensitivity.round(4))


In [ ]:
cycle_cnv_summary = (
    cnv_adata.obs.groupby(["condition", "phase"], observed=True)
    .agg(
        n_cells=("cnv_score", "size"),
        median_cnv=("cnv_score", "median"),
        cnv_high_fraction=("cnv_high", "mean"),
        tumour_derived_cnv_high_fraction=("tumour_derived_cnv_high", "mean"),
    )
    .reset_index()
)


In [ ]:
cycle_cnv_summary["normal_all_q95"] = cnv_high_threshold


In [ ]:
cycle_cnv_summary["normal_g1_q95"] = cnv_high_threshold_g1


In [ ]:
cycle_cnv_summary.to_csv(df_dir / "infercnv_cell_cycle_sensitivity.csv", index=False)


In [ ]:
display(cycle_cnv_summary.round(4))


In [ ]:
sample_population_summary = cnv_adata.obs.groupby(['sample', 'condition'], observed=True).agg(n_epithelial_cells=('tumour_derived', 'size'), n_tumour_derived=('tumour_derived', 'sum'), n_cnv_high=('cnv_high', 'sum'), n_tumour_derived_cnv_high=('tumour_derived_cnv_high', 'sum'), n_tumour_derived_cnv_high_g1=('tumour_derived_cnv_high_g1', 'sum'), median_cnv=('cnv_score', 'median')).reset_index()


In [ ]:
sample_population_summary["tumour_derived_cnv_high_fraction"] = (
    sample_population_summary["n_tumour_derived_cnv_high"]
    / sample_population_summary["n_epithelial_cells"]
)


In [ ]:
sample_population_summary["tumour_derived_cnv_high_g1_fraction"] = (
    sample_population_summary["n_tumour_derived_cnv_high_g1"]
    / sample_population_summary["n_epithelial_cells"]
)


In [ ]:
sample_population_summary.sort_values(
    ["condition", "tumour_derived_cnv_high_fraction"]
).round(3)


In [ ]:
sample_population_summary.to_csv(
    df_dir / "epithelial_population_summary_by_sample.csv", index=False
)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(
    data=cnv_adata.obs,
    x="condition",
    y="cnv_score",
    order=["N", "ER+", "HER2+", "TN"],
    showfliers=False,
    ax=ax,
)
ax.set_xlabel("")
ax.set_ylabel("inferCNV burden")
sns.despine()
save_figure(fig, fig_dir / "infercnv_score_by_condition", close=False, overwrite=False)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(
    data=cnv_adata.obs,
    x="cnv_score",
    hue="condition",
    bins=100,
    stat="density",
    common_norm=False,
    element="step",
    ax=ax,
)
ax.axvline(cnv_high_threshold, linestyle="--", linewidth=1.5, color="black")
ax.set_xlabel("inferCNV burden")
ax.set_ylabel("density")
sns.despine()
save_figure(fig, fig_dir / "infercnv_cnv_high_threshold", close=False, overwrite=False)
plt.show()


In [ ]:
plot_data = sample_population_summary[
    sample_population_summary["condition"].ne("N")
].copy()


In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
sns.barplot(
    data=plot_data,
    x="sample",
    y="tumour_derived_cnv_high_fraction",
    hue="condition",
    dodge=False,
    ax=ax,
)
ax.set_xlabel("")
ax.set_ylabel("fraction of tumour-derived epithelium that is CNV-high")
ax.tick_params(axis="x", rotation=90, labelsize=7)
sns.despine()
save_figure(fig, fig_dir / "tumour_derived_cnv_high_fraction_by_sample", close=False, overwrite=False)
plt.show()


In [ ]:
assert epi.obs_names.equals(cnv_adata.obs_names)


In [ ]:
transfer_columns = ['cnv_score', 'cnv_signal', 'tumour_derived', 'cnv_high', 'tumour_derived_cnv_high', 'g1_cell', 'tumour_derived_cnv_high_g1', 'specimen_origin']


In [ ]:
for column in transfer_columns:
    epi.obs[column] = cnv_adata.obs[column].reindex(epi.obs_names)


In [ ]:
boolean_columns = ['tumour_derived', 'cnv_high', 'tumour_derived_cnv_high', 'g1_cell', 'tumour_derived_cnv_high_g1']


In [ ]:
for column in boolean_columns:
    epi.obs[column] = epi.obs[column].astype(bool)


In [ ]:
epi.obs["specimen_origin"] = pd.Categorical(
    epi.obs["specimen_origin"], categories=origin_order
)


In [ ]:
epi.uns['epithelial_population_definitions'] = {'primary_population': 'specimen origin', 'tumour_derived': 'epithelial cells from ER+, HER2+ or TN specimens', 'normal_tissue': 'epithelial cells from normal specimens', 'tumour_derived_cnv_high': 'tumour-derived and CNV burden above normal q95', 'tumour_derived_cnv_high_limitation': 'conservative CNV-enriched subgroup; does not classify all tumour-derived cells', 'cnv_method': 'infercnvpy', 'cnv_reference': 'normal epithelial cells; leave-one-sample-out for normal samples', 'gene_annotation': 'GENCODE v50 joined by Ensembl gene ID', 'expression_layer': 'log-normalised expression', 'window_size': 100, 'step': 10, 'dynamic_threshold': float(infercnv_dynamic_threshold), 'cnv_score': 'chromosome-level RMS burden weighted by window count', 'cnv_high_normal_quantile': float(cnv_high_quantile), 'cnv_high_threshold': cnv_high_threshold, 'cnv_high_threshold_g1_reference': cnv_high_threshold_g1, 'cnv_score_vs_total_counts_r': float(r)}


In [ ]:
pd.DataFrame({'tumour_derived': [int(epi.obs['tumour_derived'].sum())], 'normal_tissue': [int((~epi.obs['tumour_derived']).sum())], 'cnv_high_all_epithelium': [int(epi.obs['cnv_high'].sum())], 'tumour_derived_cnv_high': [int(epi.obs['tumour_derived_cnv_high'].sum())], 'tumour_derived_cnv_high_g1': [int(epi.obs['tumour_derived_cnv_high_g1'].sum())]})


In [ ]:
cnv_adata.obsm["X_cnv"] = x_cnv_full


In [ ]:
cnv_file = save_dir / "epithelial_infercnv.h5ad"


In [ ]:
if cnv_file.exists():
    raise FileExistsError("CNV output already exists; use a new run_dir for a fresh analysis.")


In [ ]:
cnv_tmp_file = cnv_file.with_suffix(".h5ad.tmp")
cnv_adata.write_h5ad(cnv_tmp_file, compression="gzip")
os.replace(cnv_tmp_file, cnv_file)


In [ ]:
del cnv_adata, reference, x_cnv_full


In [ ]:
gc.collect()


## scVI integration

In [ ]:
epi_scvi = epi[:, epi.var["highly_variable"]].copy()


In [ ]:
assert epi_scvi.n_obs == epi.n_obs


In [ ]:
assert epi_scvi.n_vars == n_hvg_deviance


In [ ]:
epi_scvi.shape


In [ ]:
scvi.model.SCVI.setup_anndata(epi_scvi, layer="counts", batch_key="sample")


In [ ]:
model = scvi.model.SCVI(
    epi_scvi, n_latent=n_latent, n_layers=n_layers, gene_likelihood=gene_likelihood
)


In [ ]:
del epi_scvi


In [ ]:
gc.collect()


In [ ]:
model


In [ ]:
model.train(
    max_epochs=400,
    accelerator=scvi_accelerator,
    batch_size=1024,
    early_stopping=True,
    early_stopping_patience=20,
    early_stopping_min_delta=1.0,
)


In [ ]:
n_epochs = model.history["elbo_train"].shape[0]


In [ ]:
n_epochs


In [ ]:
model.save(str(model_dir), overwrite=False)


In [ ]:
model.history["elbo_train"].to_csv(df_dir / "scvi_epithelial_history.csv")


In [ ]:
latent = model.get_latent_representation()


In [ ]:
assert latent.shape == (epi.n_obs, 20)


In [ ]:
assert np.isfinite(latent).all()


In [ ]:
epi.obsm["X_scVI"] = latent


In [ ]:
latent.shape


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
training_history = pd.read_csv(df_dir / "scvi_epithelial_history.csv", index_col=0)
training_history.plot(ax=ax, legend=False)
ax.set_xlabel("epoch")
ax.set_ylabel("training ELBO")
sns.despine()
save_figure(fig, fig_dir / "scvi_epithelial_elbo", close=False, overwrite=False)
plt.show()


## Neighbours, UMAP and Leiden clustering

In [ ]:
sc.pp.neighbors(
    epi, use_rep="X_scVI", n_neighbors=n_neighbors, random_state=random_seed
)


In [ ]:
sc.tl.umap(epi, random_state=random_seed)


In [ ]:
leiden_summary = []


In [ ]:
for resolution in leiden_resolutions:
    key = f"leiden_r{resolution:g}"
    sc.tl.leiden(
        epi,
        resolution=resolution,
        key_added=key,
        flavor="igraph",
        n_iterations=2,
        directed=False,
        random_state=random_seed,
    )
    leiden_summary.append(
        {"resolution": resolution, "n_clusters": epi.obs[key].nunique()}
    )


In [ ]:
display(pd.DataFrame(leiden_summary))


In [ ]:
primary_key = f"leiden_r{primary_resolution:g}"


In [ ]:
resolution_stability = pd.DataFrame(
    {
        "resolution": leiden_resolutions,
        "ari_vs_primary": [
            adjusted_rand_score(
                epi.obs[primary_key], epi.obs[f"leiden_r{resolution:g}"]
            )
            for resolution in leiden_resolutions
        ],
    }
)


In [ ]:
display(resolution_stability.round(3))


In [ ]:
epi.obs["epi_leiden"] = epi.obs[primary_key].copy()


In [ ]:
epi.obs["epi_leiden"].value_counts().sort_index()


## Integration diagnostics

In [ ]:
rng = np.random.default_rng(0)


In [ ]:
n_eval = min(20000, epi.n_obs)


In [ ]:
eval_idx = rng.choice(epi.n_obs, size=n_eval, replace=False)


In [ ]:
z_eval = epi.obsm["X_scVI"][eval_idx]


In [ ]:
obs_eval = epi.obs.iloc[eval_idx]


In [ ]:
# A silhouette requires at least two manual identity categories.
bio_asw = (
    silhouette_score(z_eval, obs_eval["cl_annotation"], metric="euclidean")
    if obs_eval["cl_annotation"].nunique() > 1
    else np.nan
)

In [ ]:
batch_asw = silhouette_score(z_eval, obs_eval["sample"], metric="euclidean")


In [ ]:
print(f"annotation silhouette: {bio_asw:.3f}")


In [ ]:
print(f"sample silhouette:     {batch_asw:.3f}")


In [ ]:
pd.DataFrame(
    {
        "metric": ["annotation_silhouette", "sample_silhouette"],
        "value": [bio_asw, batch_asw],
    }
).to_csv(df_dir / "scvi_epithelial_integration_metrics.csv", index=False)


In [ ]:
cluster_composition = pd.crosstab(
    epi.obs["epi_leiden"], epi.obs["sample"], normalize="index"
)


In [ ]:
cluster_composition["top_sample_fraction"] = cluster_composition.max(axis=1)


In [ ]:
cluster_composition.to_csv(df_dir / "scvi_epithelial_cluster_sample_composition.csv")


In [ ]:
display(
    cluster_composition.sort_values("top_sample_fraction", ascending=False).round(3)
)


In [ ]:
cluster_composition.to_csv(df_dir / "scvi_epithelial_cluster_sample_composition.csv")


In [ ]:
cluster_population = epi.obs.groupby("epi_leiden", observed=True).agg(
    n_cells=("tumour_derived", "size"),
    n_tumour_derived=("tumour_derived", "sum"),
    n_tumour_derived_cnv_high=("tumour_derived_cnv_high", "sum"),
    n_tumour_derived_cnv_high_g1=("tumour_derived_cnv_high_g1", "sum"),
    median_cnv=("cnv_score", "median"),
)


In [ ]:
cluster_population["tumour_derived_fraction"] = (
    cluster_population["n_tumour_derived"] / cluster_population["n_cells"]
)


In [ ]:
cluster_population["tumour_derived_cnv_high_fraction"] = (
    cluster_population["n_tumour_derived_cnv_high"] / cluster_population["n_cells"]
)


In [ ]:
cluster_population.sort_values(
    "tumour_derived_cnv_high_fraction", ascending=False
).round(3)


In [ ]:
cluster_population.to_csv(df_dir / "scvi_epithelial_cluster_population_composition.csv")


In [ ]:
marker_genes = [
    g
    for g in [
        "EPCAM",
        "KRT8",
        "KRT18",
        "KRT19",
        "KRT7",
        "KRT5",
        "KRT14",
        "KRT17",
        "MUC1",
    ]
    if g in epi.var_names
]


## Epithelial UMAPs

In [ ]:
condition_palette = {
    "N": "#031A6B",
    "ER+": "#F46036",
    "HER2+": "#2CA02C",
    "TN": "#976391",
}


In [ ]:
condition_order = ["TN", "HER2+", "ER+", "N"]


In [ ]:
epi.obs["condition"] = pd.Categorical(
    epi.obs["condition"].astype(str), categories=condition_order, ordered=True
)


In [ ]:
epi.uns["condition_colors"] = [condition_palette[c] for c in condition_order]


In [ ]:
epi.obs["cl_annotation"] = epi.obs["cl_annotation"].astype("category")


In [ ]:
epi.obs["sample_short"] = (
    epi.obs["sample_short"].astype("category").cat.remove_unused_categories()
)


In [ ]:
draw_order = ["N", "TN", "HER2+", "ER+"]


In [ ]:
rank = {condition: i for i, condition in enumerate(draw_order)}


In [ ]:
plot_idx = epi.obs["condition"].astype(str).map(rank).sort_values().index


In [ ]:
sc.pl.umap(epi, color="epi_leiden", palette="tab20b", legend_loc="on data", show=False)
save_umap_figure(plt.gcf(), fig_dir / "epithelial_umap_cluster_numbers", close=False, overwrite=False)
plt.show()


In [ ]:
sc.pl.umap(
    epi[plot_idx],
    color="condition",
    palette=condition_palette,
    frameon=True,
    show=False,
)
ax = plt.gca()
ax.spines[["top", "right"]].set_visible(False)
ax.set_xlabel("UMAP1", fontsize=25)
ax.set_ylabel("UMAP2", fontsize=25)
ax.tick_params(axis="both", labelsize=12)
legend = ax.get_legend()
if legend is not None:
    plt.setp(legend.get_texts(), fontsize=15)
    for handle in legend.legend_handles:
        handle.set_sizes([70])
save_umap_figure(plt.gcf(), fig_dir / "epithelial_umap_condition", close=False, overwrite=False)
plt.show()


In [ ]:
sc.pl.umap(epi, color="cl_annotation", frameon=False, show=False)
save_umap_figure(plt.gcf(), fig_dir / "epithelial_umap_cl_annotation", close=False, overwrite=False)
plt.show()


In [ ]:
sc.tl.rank_genes_groups(
    epi,
    groupby="epi_leiden",
    layer="lognorm",
    use_raw=False,
    method="wilcoxon",
    pts=True,
    tie_correct=True,
)


In [ ]:
cluster_markers = sc.get.rank_genes_groups_df(epi, group=None)


In [ ]:
cluster_markers.to_csv(df_dir / "epithelial_cluster_markers_wilcoxon.csv", index=False)


In [ ]:
# Compare epithelial clusters with the current manual atlas labels.
generic_label = "Epithelial"
label_overlap = pd.crosstab(
    epi.obs["epi_leiden"], epi.obs["cl_annotation"], normalize="index"
).mul(100).round(1)
named = [label for label in label_overlap.columns if label != generic_label]

In [ ]:
# Percentages use all cells in each epithelial cluster as the denominator.
# A missing generic category has zero cells; absent named labels remain undefined.
cluster_identity = pd.DataFrame({
    "n_cells": epi.obs["epi_leiden"].value_counts().reindex(label_overlap.index),
    "pct_generic": label_overlap.get(
        generic_label, pd.Series(0.0, index=label_overlap.index)
    ),
    "dominant_named": label_overlap[named].idxmax(axis=1) if named else pd.NA,
    "pct_dominant_named": label_overlap[named].max(axis=1) if named else 0.0,
})
cluster_identity["resolves_generic"] = (
    cluster_identity["pct_generic"].ge(20)
    & cluster_identity["pct_dominant_named"].ge(50)
)

In [ ]:
# Export the overlap and marker evidence without changing cell labels.
top_markers = (
    cluster_markers.loc[cluster_markers["pvals_adj"] < 0.05]
    .sort_values("scores", ascending=False)
    .groupby("group", observed=True)["names"]
    .apply(lambda genes: ", ".join(genes.head(5)))
)
cluster_identity["top_markers"] = top_markers.reindex(cluster_identity.index)
cluster_identity.to_csv(df_dir / "epithelial_cluster_vs_atlas_annotation.csv")
label_overlap.to_csv(df_dir / "epithelial_cluster_annotation_overlap.csv")

In [ ]:
generic_cells = epi.obs["cl_annotation"].eq(generic_label)
generic_spread = (
    epi.obs.loc[generic_cells, "epi_leiden"].value_counts().rename("n_generic_cells")
)
generic_spread = generic_spread[generic_spread > 0]
print(f"{int(generic_cells.sum()):,} cells labelled '{generic_label}' in 002 "
      f"spread over {len(generic_spread)} of {epi.obs['epi_leiden'].nunique()} new clusters")
print(f"Clusters with >50% generic labels: "
      f"{int(cluster_identity['pct_generic'].gt(50).sum())}")
print(f"Clusters with generic cells and a named majority: "
      f"{int(cluster_identity['resolves_generic'].sum())}")
display(cluster_identity.sort_values("pct_generic", ascending=False))

In [ ]:
# Are the unresolved clusters a cell state, or a quality artefact?
cluster_quality = epi.obs.groupby("epi_leiden", observed=True).agg(
    n_cells=("total_counts", "size"),
    median_counts=("total_counts", "median"),       # low => low-complexity cells
    median_genes=("n_genes_by_counts", "median"),
    median_pct_mt=("pct_counts_mt", "median"),      # high => stressed or dying
    median_pct_ribo=("pct_counts_ribo", "median"),  # high => housekeeping-dominated
    median_top20_pct=("pct_counts_in_top_20_genes", "median"),  # high => low diversity
    median_doublet=("doublet_score", "median"),
    pct_tumour_derived=("tumour_derived", "mean"),
    median_cnv=("cnv_score", "median"),
).round(3)
cluster_quality["pct_tumour_derived"] = (100 * cluster_quality["pct_tumour_derived"]).round(1)
cluster_quality.to_csv(df_dir / "epithelial_cluster_quality.csv")
display(cluster_quality.sort_values("median_genes"))


In [ ]:
sc.tl.dendrogram(epi, groupby="epi_leiden", use_rep="X_scVI")

marker_dotplot = sc.pl.rank_genes_groups_dotplot(
    epi, groupby="epi_leiden", n_genes=5,
    values_to_plot="logfoldchanges", standard_scale="var",
    figsize=(24, 8), return_fig=True, show=False,
)
# The colour scale is rescaled separately for each gene.
marker_dotplot.legend(
    width=3.2, size_title="Cells expressing\ngene (%)",
    colorbar_title="Gene-wise scaled\nlog fold change",
)
marker_dotplot.make_figure()
save_figure(marker_dotplot.fig, fig_dir / "epithelial_cluster_markers_top5", close=False, overwrite=False)
plt.show()


In [ ]:
sc.pl.umap(epi, color="sample_short", legend_loc=None, frameon=False, show=False)
ax = plt.gca()
sample_to_color = dict(
    zip(epi.obs["sample_short"].cat.categories, epi.uns["sample_short_colors"])
)
handles = []
for condition in condition_order:
    handles.append(Line2D([], [], linestyle="none", label=condition, color="black"))
    for sample in sorted(
        epi.obs.loc[epi.obs["condition"] == condition, "sample_short"].unique()
    ):
        handles.append(
            Line2D(
                [],
                [],
                marker="o",
                linestyle="none",
                markersize=6,
                markerfacecolor=sample_to_color[sample],
                markeredgecolor="none",
                label=f"  {sample}",
            )
        )
ax.legend(
    handles=handles,
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False,
    fontsize=7,
)
save_umap_figure(plt.gcf(), fig_dir / "epithelial_umap_sample", close=False, overwrite=False)
plt.show()


In [ ]:
umap = epi.obsm["X_umap"]


In [ ]:
tumour_derived_cnv_high_plot = epi.obs["tumour_derived_cnv_high"].to_numpy()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
sc.pl.umap(
    epi,
    color="cnv_score",
    color_map="viridis",
    vmax="p99.9",
    frameon=False,
    title="CNV burden",
    ax=axes[0],
    show=False,
)
axes[1].scatter(
    umap[:, 0], umap[:, 1], s=1, color="#D0D0D0", linewidths=0, rasterized=True
)
axes[1].scatter(
    umap[tumour_derived_cnv_high_plot, 0],
    umap[tumour_derived_cnv_high_plot, 1],
    s=1,
    color="#D55E00",
    linewidths=0,
    rasterized=True,
)
axes[1].set_title("CNV-high tumour-derived epithelium")
axes[1].set_xticks([])
axes[1].set_yticks([])
axes[1].set_aspect("equal", adjustable="datalim")
axes[1].set_frame_on(False)
axes[1].legend(
    handles=[
        Line2D(
            [],
            [],
            marker="o",
            linestyle="none",
            color="#D0D0D0",
            label="other epithelium",
        ),
        Line2D(
            [],
            [],
            marker="o",
            linestyle="none",
            color="#D55E00",
            label="CNV-high tumour-derived",
        ),
    ],
    frameon=False,
    loc="lower left",
    markerscale=3,
    fontsize=8,
)
sc.pl.umap(
    epi,
    color="epi_leiden",
    legend_loc="right margin",
    frameon=False,
    title="Epithelial clusters",
    ax=axes[2],
    show=False,
)
fig.tight_layout()
save_umap_figure(fig, fig_dir / "epithelial_umap_population_and_cnv", close=False, overwrite=False)
plt.show()


In [ ]:
def save_cnv_panels(epi, output_dir):
    """Standalone CNV panels for figure assembly, using the saved embedding."""
    output_dir = Path(output_dir)
    fig = sc.pl.umap(
        epi,
        color="cnv_score",
        cmap="viridis",
        vmax="p99.9",
        frameon=False,
        title="CNV burden",
        return_fig=True,
        show=False,
    )
    save_umap_figure(
        fig, output_dir / "epithelial_umap_cnv_burden", close=False, overwrite=False
    )
    plt.show()
    coordinates = epi.obsm["X_umap"]
    high = epi.obs["tumour_derived_cnv_high"].to_numpy(bool)
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(*coordinates.T, s=1, color="#D0D0D0", linewidths=0, rasterized=True)
    ax.scatter(
        *coordinates[high].T, s=1, color="#D55E00", linewidths=0, rasterized=True
    )
    ax.set_title("CNV-high tumour-derived epithelium", loc="left")
    ax.set_axis_off()
    ax.legend(
        handles=[
            Line2D(
                [],
                [],
                marker="o",
                linestyle="none",
                color="#D55E00",
                label="tumour-derived, above normal epithelial q95",
            ),
            Line2D(
                [],
                [],
                marker="o",
                linestyle="none",
                color="#D0D0D0",
                label="other epithelial cells",
            ),
        ],
        frameon=False,
        fontsize=8,
        loc="lower left",
    )
    save_umap_figure(
        fig, output_dir / "epithelial_umap_cnv_high", close=False, overwrite=False
    )
    plt.show()


In [ ]:
save_cnv_panels(epi, fig_dir)


In [ ]:
origin_palette = {
    "normal-tissue epithelium": "#56B4E9",
    "tumour-derived epithelium": "#D55E00",
}


In [ ]:
sc.pl.umap(
    epi,
    color="specimen_origin",
    palette=origin_palette,
    frameon=False,
    show=False,
    title="Epithelial cells by specimen origin",
)
save_umap_figure(plt.gcf(), fig_dir / "epithelial_umap_specimen_origin", close=False, overwrite=False)
plt.show()


## Feature-set sensitivity

The primary scVI model uses only the deviance-ranked genes. A second model adds
the antigen and meta-programme genes to test whether that targeted feature choice changes
the neighbourhood or cluster structure beyond ordinary run-to-run variation.

In [ ]:
epi_protected = epi[:, epi.var["highly_variable_protected"]].copy()


In [ ]:
assert epi_protected.n_vars == n_hvg_protected


In [ ]:
scvi.model.SCVI.setup_anndata(epi_protected, layer="counts", batch_key="sample")


In [ ]:
model_protected = scvi.model.SCVI(
    epi_protected, n_latent=n_latent, n_layers=n_layers, gene_likelihood=gene_likelihood
)


In [ ]:
model_protected.train(
    max_epochs=400, accelerator=scvi_accelerator, batch_size=1024,
    early_stopping=True, early_stopping_patience=20, early_stopping_min_delta=1.0)


In [ ]:
epi.obsm["X_scVI_protected"] = model_protected.get_latent_representation()


In [ ]:
model_protected.save(str(models_dir / "scvi_epithelial_protected_model"), overwrite=False)


In [ ]:
del model_protected


In [ ]:
gc.collect()


In [ ]:
sc.pp.neighbors(
    epi,
    use_rep="X_scVI_protected",
    n_neighbors=n_neighbors,
    random_state=random_seed,
    key_added="protected_neighbors",
)


In [ ]:
sc.tl.leiden(
    epi,
    resolution=primary_resolution,
    key_added="epi_leiden_protected",
    neighbors_key="protected_neighbors",
    flavor="igraph",
    n_iterations=2,
    directed=False,
    random_state=random_seed,
)


In [ ]:
del epi_protected


In [ ]:
gc.collect()


In [ ]:
epi_seed = epi[:, epi.var["highly_variable"]].copy()


In [ ]:
scvi.settings.seed = random_seed + 1


In [ ]:
scvi.model.SCVI.setup_anndata(epi_seed, layer="counts", batch_key="sample")


In [ ]:
model_seed = scvi.model.SCVI(
    epi_seed, n_latent=n_latent, n_layers=n_layers, gene_likelihood=gene_likelihood
)


In [ ]:
model_seed.train(
    max_epochs=400,
    accelerator=scvi_accelerator,
    batch_size=1024,
    early_stopping=True,
    early_stopping_patience=20,
    early_stopping_min_delta=1.0,
)

In [ ]:
epi.obsm["X_scVI_seed2"] = model_seed.get_latent_representation()


In [ ]:
model_seed.save(str(models_dir / "scvi_epithelial_seed2_model"), overwrite=False)


In [ ]:
scvi.settings.seed = random_seed


In [ ]:
sc.pp.neighbors(
    epi,
    use_rep="X_scVI_seed2",
    n_neighbors=n_neighbors,
    random_state=random_seed,
    key_added="seed2_neighbors",
)


In [ ]:
sc.tl.leiden(
    epi,
    resolution=primary_resolution,
    key_added="epi_leiden_seed2",
    neighbors_key="seed2_neighbors",
    flavor="igraph",
    n_iterations=2,
    directed=False,
    random_state=random_seed,
)


In [ ]:
del epi_seed, model_seed


In [ ]:
gc.collect()


In [ ]:
k_overlap = n_neighbors


In [ ]:
overlap_idx = eval_idx


In [ ]:
def knn_indices(representation):
    values = epi.obsm[representation][overlap_idx]
    model_nn = NearestNeighbors(n_neighbors=k_overlap + 1).fit(values)
    return model_nn.kneighbors(values, return_distance=False)[:, 1:]


In [ ]:
primary_neighbors = knn_indices("X_scVI")


In [ ]:
comparisons = [
    ("protected feature set", "X_scVI_protected", "epi_leiden_protected"),
    ("different seed", "X_scVI_seed2", "epi_leiden_seed2"),
]


In [ ]:
rows = []


In [ ]:
for label, representation, clustering in comparisons:
    other_neighbors = knn_indices(representation)
    overlap = np.array(
        [
            len(set(left) & set(right)) / k_overlap
            for left, right in zip(primary_neighbors, other_neighbors)
        ]
    )
    rows.append(
        {
            "comparison": label,
            "leiden_ari": adjusted_rand_score(
                epi.obs["epi_leiden"], epi.obs[clustering]
            ),
            "leiden_nmi": normalized_mutual_info_score(
                epi.obs["epi_leiden"], epi.obs[clustering]
            ),
            "knn_overlap_mean": overlap.mean(),
            "knn_overlap_median": np.median(overlap),
            "n_clusters": epi.obs[clustering].nunique(),
        }
    )


In [ ]:
latent_comparison = pd.DataFrame(rows)


In [ ]:
latent_comparison.to_csv(df_dir / "scvi_latent_comparison.csv", index=False)


In [ ]:
display(latent_comparison.round(3))


In [ ]:
rows = []


In [ ]:
for label, representation in [
    ("primary_deviance", "X_scVI"),
    ("protected_sensitivity", "X_scVI_protected"),
    ("primary_second_seed", "X_scVI_seed2"),
]:
    values = epi.obsm[representation][eval_idx]
    rows.append(
        {
            "model": label,
            "annotation_silhouette": (
                silhouette_score(values, obs_eval["cl_annotation"])
                if obs_eval["cl_annotation"].nunique() > 1 else np.nan
            ),
            "sample_silhouette": silhouette_score(values, obs_eval["sample"]),
            "condition_silhouette": silhouette_score(
                values, obs_eval["condition"].astype(str)
            ),
        }
    )


In [ ]:
model_comparison = pd.DataFrame(rows)


In [ ]:
display(model_comparison.round(3))


In [ ]:
del model


In [ ]:
gc.collect()


In [ ]:
rows = []


In [ ]:
for annotation in obs_eval["cl_annotation"].value_counts().index:
    mask = obs_eval["cl_annotation"].eq(annotation).to_numpy()
    if mask.sum() < 100 or obs_eval.loc[mask, "sample"].nunique() < 2:
        continue
    row = {"cl_annotation": annotation, "n_cells": int(mask.sum())}
    for label, representation in [
        ("primary_deviance", "X_scVI"),
        ("protected_sensitivity", "X_scVI_protected"),
        ("primary_second_seed", "X_scVI_seed2"),
    ]:
        row[label] = silhouette_score(
            epi.obsm[representation][eval_idx][mask], obs_eval.loc[mask, "sample"]
        )
    rows.append(row)


In [ ]:
within_annotation = pd.DataFrame(rows)


In [ ]:
display(within_annotation.round(3))


In [ ]:
cluster_overlap = pd.crosstab(epi.obs["epi_leiden"], epi.obs["epi_leiden_protected"])


In [ ]:
best_match = cluster_overlap.max(axis=1) / cluster_overlap.sum(axis=1)


In [ ]:
feature_effect = latent_comparison.iloc[0]


In [ ]:
seed_floor = latent_comparison.iloc[1]


In [ ]:
epi.uns["protected_gene_sensitivity"] = {
    "primary_features": int(n_hvg_deviance),
    "protected_features": int(n_hvg_protected),
    "protected_added": int(n_protected_added),
    "feature_set_ari": float(feature_effect["leiden_ari"]),
    "seed_floor_ari": float(seed_floor["leiden_ari"]),
    "feature_set_knn_overlap": float(feature_effect["knn_overlap_mean"]),
    "seed_floor_knn_overlap": float(seed_floor["knn_overlap_mean"]),
    "k": int(k_overlap),
}


In [ ]:
model_comparison.to_csv(df_dir / "scvi_feature_sensitivity.csv", index=False)


In [ ]:
within_annotation.to_csv(
    df_dir / "scvi_feature_sensitivity_by_annotation.csv", index=False
)


In [ ]:
best_match.rename("largest_matching_fraction").to_csv(
    df_dir / "scvi_feature_cluster_correspondence.csv"
)


## Save

In [ ]:
final_positive = {
    gene: int((np.asarray(epi[:, gene].layers["counts"].todense()).ravel() > 0).sum())
    for gene in target_genes
}


In [ ]:
assert epi.n_obs == n_cells_start


In [ ]:
assert epi.n_vars == n_genes_start


In [ ]:
assert final_positive == {gene: baseline_positive[gene] for gene in final_positive}


In [ ]:
epi.uns["epithelial_integration"] = {
    "hvg_method": "binomial deviance (scry), maximum across samples",
    "primary_feature_set": "deviance-ranked genes only",
    "n_hvg_primary": int(n_hvg_deviance),
    "n_hvg_protected_sensitivity": int(n_hvg_protected),
    "scvi_n_latent": int(n_latent),
    "scvi_n_layers": int(n_layers),
    "scvi_gene_likelihood": gene_likelihood,
    "scvi_batch_key": "sample",
    "scvi_epochs": int(n_epochs),
    "n_neighbors": int(n_neighbors),
    "leiden_resolutions": [float(value) for value in leiden_resolutions],
    "leiden_primary": float(primary_resolution),
    "seed": int(random_seed),
}


In [ ]:
out_path = save_dir / "epithelial_tumour_scvi.h5ad"


In [ ]:
for name in list(epi.layers.keys()):
    if not sp.issparse(epi.layers[name]):
        epi.layers[name] = sp.csr_matrix(epi.layers[name])


In [ ]:
if not sp.issparse(epi.X):
    epi.X = sp.csr_matrix(epi.X)


In [ ]:
tmp_path = out_path.with_suffix(".h5ad.tmp")


In [ ]:
if out_path.exists():
    raise FileExistsError("Analysis output already exists; choose a new run_dir for a fresh analysis.")


In [ ]:
epi.write_h5ad(tmp_path, compression="gzip")


In [ ]:
os.replace(tmp_path, out_path)


In [ ]:
display(pd.DataFrame({"saved": [out_path.name]}))


In [ ]:
check = sc.read_h5ad(out_path, backed="r")


In [ ]:
assert check.n_obs == epi.n_obs


In [ ]:
assert check.n_vars == epi.n_vars


In [ ]:
check.shape


In [ ]:
del check


In [ ]:
gc.collect()


In [ ]:
session_info.show()
